# HKU-FYP Master Pipeline v1.1 (Benchmark + Threshold + Confusion Matrix)

This notebook extends v1 with:
- Naive baselines (always-up, class-frequency random)
- Confusion matrix and class distribution
- Validation threshold tuning for probabilistic models


In [ ]:
# !pip install yfinance pandas numpy scikit-learn matplotlib seaborn xgboost
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, precision_score, recall_score
)


In [ ]:
TICKERS = [
    'AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','JPM','V','MA',
    'UNH','HD','PG','JNJ','XOM','AVGO','COST','KO','PEP','MRK'
]
START_DATE = '2005-01-01'
END_DATE = None
HORIZONS = [1,3,5,10]
FEATURES = ['ret_1d','ret_5d','ret_10d','vol_10d','vol_20d','ma_ratio_5_20','mom_10']
SEED = 42


In [ ]:
def download_ohlcv(tickers, start=START_DATE, end=END_DATE):
    frames = []
    for t in tickers:
        df = yf.download(t, start=start, end=end, auto_adjust=True, progress=False)
        if df is None or df.empty:
            continue
        df = df[['Open','High','Low','Close','Volume']].copy()
        df.columns = ['open','high','low','close','volume']
        df['ticker'] = t
        df['date'] = df.index
        frames.append(df.reset_index(drop=True))
    if not frames:
        raise ValueError('No data downloaded.')
    return pd.concat(frames, ignore_index=True)


def add_technical_features(df):
    out = []
    for t, g in df.groupby('ticker'):
        g = g.sort_values('date').copy()
        g['ret_1d'] = g['close'].pct_change(1)
        g['ret_5d'] = g['close'].pct_change(5)
        g['ret_10d'] = g['close'].pct_change(10)
        g['vol_10d'] = g['ret_1d'].rolling(10).std()
        g['vol_20d'] = g['ret_1d'].rolling(20).std()
        g['ma_5'] = g['close'].rolling(5).mean()
        g['ma_20'] = g['close'].rolling(20).mean()
        g['ma_ratio_5_20'] = g['ma_5'] / g['ma_20']
        g['mom_10'] = g['close'] / g['close'].shift(10) - 1
        out.append(g)
    return pd.concat(out, ignore_index=True)


def add_labels(df, horizons=HORIZONS):
    out = []
    for t, g in df.groupby('ticker'):
        g = g.sort_values('date').copy()
        for h in horizons:
            fwd_ret = g['close'].shift(-h) / g['close'] - 1
            g[f'label_{h}d'] = (fwd_ret > 0).astype(int)
        out.append(g)
    return pd.concat(out, ignore_index=True)


def time_split(df, train_ratio=0.7, val_ratio=0.15):
    dates = np.sort(df['date'].unique())
    n = len(dates)
    d1 = dates[int(n*train_ratio)]
    d2 = dates[int(n*(train_ratio+val_ratio))]
    train = df[df['date'] < d1]
    val = df[(df['date'] >= d1) & (df['date'] < d2)]
    test = df[df['date'] >= d2]
    return train, val, test


In [ ]:
def get_models(random_state=SEED):
    models = {
        'logreg': LogisticRegression(max_iter=1000),
        'rf': RandomForestClassifier(n_estimators=250, random_state=random_state, n_jobs=-1),
    }
    try:
        from xgboost import XGBClassifier
        models['xgb'] = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=random_state,
            n_jobs=-1
        )
    except Exception:
        models['gb_fallback'] = GradientBoostingClassifier(random_state=random_state)
    return models


def calc_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_up': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_up': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_up': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_down': precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        'recall_down': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def tune_threshold(y_true, y_prob, objective='balanced_accuracy'):
    thresholds = np.arange(0.35, 0.66, 0.01)
    best_t, best_s = 0.5, -1
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        if objective == 'f1_up':
            s = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        else:
            s = balanced_accuracy_score(y_true, y_pred)
        if s > best_s:
            best_s, best_t = s, float(t)
    return best_t, best_s


In [ ]:
def evaluate_one_horizon(df, horizon=1, threshold_objective='balanced_accuracy'):
    label_col = f'label_{horizon}d'
    use = df[['date','ticker'] + FEATURES + [label_col]].dropna().copy()
    train, val, test = time_split(use)

    X_train, y_train = train[FEATURES], train[label_col].astype(int)
    X_val, y_val = val[FEATURES], val[label_col].astype(int)
    X_test, y_test = test[FEATURES], test[label_col].astype(int)

    rows = []

    # Class distribution summary
    up_train = float(y_train.mean())
    up_test = float(y_test.mean())

    # Naive baseline 1: always-up
    pred_up = np.ones(len(y_test), dtype=int)
    m = calc_metrics(y_test, pred_up)
    rows.append({'horizon': f'{horizon}D', 'model': 'baseline_always_up', 'threshold': 1.0,
                 'train_up_rate': up_train, 'test_up_rate': up_test, 'test_samples': len(y_test), **m})

    # Naive baseline 2: random by train up-rate
    rng = np.random.default_rng(SEED)
    pred_rand = (rng.random(len(y_test)) < up_train).astype(int)
    m = calc_metrics(y_test, pred_rand)
    rows.append({'horizon': f'{horizon}D', 'model': 'baseline_random_freq', 'threshold': up_train,
                 'train_up_rate': up_train, 'test_up_rate': up_test, 'test_samples': len(y_test), **m})

    # ML models
    for name, model in get_models().items():
        model.fit(X_train, y_train)

        # threshold tuning only for probabilistic models
        t_star = 0.5
        if hasattr(model, 'predict_proba'):
            val_prob = model.predict_proba(X_val)[:,1]
            t_star, _ = tune_threshold(y_val.values, val_prob, objective=threshold_objective)
            test_prob = model.predict_proba(X_test)[:,1]
            pred = (test_prob >= t_star).astype(int)
        else:
            pred = model.predict(X_test)

        m = calc_metrics(y_test, pred)
        rows.append({'horizon': f'{horizon}D', 'model': name, 'threshold': t_star,
                     'train_up_rate': up_train, 'test_up_rate': up_test, 'test_samples': len(y_test), **m})

    return pd.DataFrame(rows)


In [ ]:
raw = download_ohlcv(TICKERS)
feat = add_technical_features(raw)
all_df = add_labels(feat)

results = []
for h in HORIZONS:
    results.append(evaluate_one_horizon(all_df, horizon=h, threshold_objective='balanced_accuracy'))

results_df = pd.concat(results, ignore_index=True)
results_df = results_df.sort_values(['horizon','balanced_accuracy'], ascending=[True, False])
results_df


In [ ]:
os.makedirs('../outputs/metrics', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

results_df.to_csv('../outputs/metrics/pilot_v1_1_metrics.csv', index=False)
summary_cols = ['horizon','model','threshold','accuracy','balanced_accuracy','f1_up','precision_up','recall_up','precision_down','recall_down','train_up_rate','test_up_rate','test_samples']
results_df[summary_cols].to_csv('../outputs/tables/pilot_v1_1_summary_table.csv', index=False)
print('Saved: ../outputs/metrics/pilot_v1_1_metrics.csv')
print('Saved: ../outputs/tables/pilot_v1_1_summary_table.csv')


## How to read this output
- If a model cannot beat `baseline_always_up` on **balanced_accuracy**, the signal is weak.
- `train_up_rate` and `test_up_rate` explain market drift/imbalance.
- Use confusion matrix fields (`tn/fp/fn/tp`) and per-class recall to avoid being fooled by headline accuracy.
